# GAMSPy model example

## Introduction

This quick start guide is designed to provide you with a concise overview of GAMSPy and its key features. By the end of this guide, you'll have a solid understanding of how to create basic mathematical models using GAMSPy. For more advanced features, we recommend exploring our comprehensive [user guide](https://gamspy.readthedocs.io/en/latest/user/index.html) and the extensive [model library](https://gamspy.readthedocs.io/en/latest/user/model_library.html).

While not mandatory, having a basic understanding of Python programming and familiarity with the [Pandas library](https://pandas.pydata.org/). will be helpful in following this tutorial.

## A Transportation Problem

In this guide, we'll delve into an example of the transportation problem. This classic scenario involves managing supplies from various plants to meet demands at multiple markets for a single commodity. Additionally, we have the unit costs associated with shipping the commodity from plants to markets. The fundamental economic question here is: 

> How can we optimize the shipment quantities between each plant and market to minimize the total transport cost?

The problem's algebraic representation typically takes the following format:

Indices (Sets):

- $i$ = plants
- $j$ = markets

Given Data (Parameters):

- $a_i$ = supply of commodity at plant $i$ (in cases)
- $b_j$ = demand for commodity at market $j$ (in cases)
- $c_{ij}$ = cost per unit of shipment between plant $i$ and market $j$

Decision Variables:

- $x_{ij}$ = amount of commodity to ship from plant $i$ to market $j$ where $x_{ij} \ge 0$ for all $i,j$

Constraints:

- Observe supply limit at plant $i$: $\sum_j x_{ij} \le a_i \: \forall i$
- Satisfy demand at market $j$: $\sum_i x_{ij} \ge b_j \: \forall j$
- Objective Function: Minimize $\sum_i \sum_j c_{ij} \cdot x_{ij}$

In [ ]:
import subprocess
import html
from IPython.display import display, HTML

def check_gpu():
    try:
        result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=5)
        result.check_returncode()
        lines = result.stdout.splitlines()
        gpu_info = lines[2] if len(lines) > 2 else "GPU detected"
        gpu_info_escaped = html.escape(gpu_info)
        display(HTML(f"""
        <div style="border:2px solid #4CAF50;padding:10px;border-radius:10px;background:#e8f5e9;">
            <h3>✅ GPU is enabled</h3>
            <pre>{gpu_info_escaped}</pre>
        </div>
        """))
        return True
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired, FileNotFoundError, IndexError) as e:
        display(HTML("""
        <div style="border:2px solid red;padding:15px;border-radius:10px;background:#ffeeee;">
            <h3>⚠️ GPU not detected!</h3>
            <p>This notebook requires a <b>GPU runtime</b>.</p>
            
            <h4>If running in Google Colab:</h4>
            <ol>
              <li>Click on <b>Runtime → Change runtime type</b></li>
              <li>Set <b>Hardware accelerator</b> to <b>GPU</b></li>
              <li>Then click <b>Save</b> and <b>Runtime → Restart runtime</b>.</li>
            </ol>
            
            <h4>If running in Docker:</h4>
            <ol>
              <li>Ensure you have <b>NVIDIA Docker runtime</b> installed (<code>nvidia-docker2</code>)</li>
              <li>Run container with GPU support: <code>docker run --gpus all ...</code></li>
              <li>Or use: <code>docker run --runtime=nvidia ...</code> for older Docker versions</li>
              <li>Verify GPU access: <code>docker run --gpus all nvidia/cuda:12.0.0-base-ubuntu22.04 nvidia-smi</code></li>
            </ol>
            
            <p><b>Additional resources:</b></p>
            <ul>
              <li><a href="https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/install-guide.html" target="_blank">NVIDIA Container Toolkit Installation Guide</a></li>
            </ul>
        </div>
        """))
        return False

check_gpu()

In [ ]:
# Enable this in case you are running this in google colab or such places where cuOpt is not yet installed
#!pip uninstall -y cuda-python cuda-bindings cuda-core
#!pip install --upgrade --extra-index-url=https://pypi.nvidia.com cuopt-cu12 nvidia-nvjitlink-cu12 rapids-logger==0.1.19
#!pip install --upgrade --extra-index-url=https://pypi.nvidia.com cuopt-cu13 nvidia-nvjitlink-cu13 rapids-logger==0.1.1

In [ ]:
# Install other dependencies (if not already installed)

!pip install -q gamspy
import subprocess
import sys
# Change the zip URL below depending on your target CUDA version and architecture (x86_64 or aarch64).
# See https://github.com/GAMS-dev/cuoptlink-builder/releases for available builds.
!wget -nc -nv -O cuopt-link-release.zip "https://github.com/GAMS-dev/cuoptlink-builder/releases/download/v0.0.5e/cuopt-link-release-cu13-x86_64.zip"
gams_base_path = subprocess.check_output([sys.executable, '-m', 'gamspy', 'show', 'base']).decode('utf-8').strip()
subprocess.run(f"unzip -o cuopt-link-release.zip -d {gams_base_path}", shell=True, check=True)

## Data

Before we dive into the optimization process, let's handle our data using the [Pandas library](https://pandas.pydata.org/). We'll begin by organizing the necessary information, which we will subsequently feed into our optimization model.

In [ ]:
import pandas as pd

capacities = pd.DataFrame(
    [["seattle", 350], ["san-diego", 600]], columns=["city", "capacity"]
).set_index("city")

capacities

In [ ]:
demands = pd.DataFrame(
    [["new-york", 325], ["chicago", 300], ["topeka", 275]], columns=["city", "demand"]
).set_index("city")

demands

In [ ]:
distances = pd.DataFrame(
    [
        ["seattle", "new-york", 2.5],
        ["seattle", "chicago", 1.7],
        ["seattle", "topeka", 1.8],
        ["san-diego", "new-york", 2.5],
        ["san-diego", "chicago", 1.8],
        ["san-diego", "topeka", 1.4],
    ],
    columns=["from", "to", "distance"],
).set_index(["from", "to"])

distances

In [ ]:
freight_cost = 90

## Symbol Declaration

In line with our systematic breakdown of the transportation problem into sets, parameters, variables, and constraints, we will adopt a similar approach to define the problem as a GAMSPy `Model`. To do so, it is essential to import the `gamspy` library initially.

In [ ]:
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense

### Container

Before we proceed further, let's create a `Container` to encapsulate all the relevant information for our GAMSPy ``Model``. This ``Container`` acts as a centralized hub, gathering essential data, sets, parameters, variables, and constraints, providing a clear structure for our optimization problem.

In [ ]:
m = Container()

### Sets

Sets serve as the fundamental building blocks of a GAMSPy ``Model``, directly corresponding to the indices in the algebraic representations of models. In our transportation problem context, we have defined the following indices:

- $i$ = plants
- $j$ = markets

For detailed guidance on using sets, please refer to the [set section](https://gamspy.readthedocs.io/en/latest/user/basics/set.html) of our user guide.

There a two ways to declare sets:

1. Separate declaration and data assignment
2. Combine declaration and data assignment

#### Separate declaration and data assignment


In [ ]:
i = Set(container=m, name="i", description="plants")
i.setRecords(capacities.index)

#### Combine declaration and data assignment


In [ ]:
j = Set(container=m, name="j", description="markets", records=demands.index)

The effect of using the above `Set` statements is that we declared two sets, namely $i$ and $j$. Additionally, we provided descriptions to elaborate on their meaning, enhancing the readability of our ``Model``. Lastly, we assigned members to the sets, establishing a clear connection between the abstract sets and their real-world counterparts.

$i$ = {Seattle, San Diego}

$j$ = {New York, Chicago, Topeka}

To verify the content of a set, you can use `<set name>.records`.

In [ ]:
i.records

In [ ]:
j.records

### Parameters

Declaring parameters involves using `Parameter`. Each parameter is assigned a name and a description.  Note that parameter $a_i$ is indexed by $i$. To accommodate these indices, we include the `domain` attribute, pointing to the corresponding set.

It is worth mentioning that, similar to sets, you have the flexibility to either combine or separate the declaration and data assignment steps. For convenience, we will proceed by combining the declaration and data assignment.


In [ ]:
a = Parameter(
    container=m, 
    name="a",
    domain=i,
    description="supply of commodity at plant i (in cases)",
    records=capacities.reset_index(),
)
a.records

In [ ]:
b = Parameter(
    container=m,
    name="b",
    domain=j,
    description="demand for commodity at market j (in cases)",
    records=demands.reset_index(),
)
b.records

In [ ]:
c = Parameter(
    container=m,
    name="c",
    domain=[i, j],
    description="cost per unit of shipment between plant i and market j",
)

The cost per unit of shipment between plant $i$ and market $j$ is derived from the distance between $i$ and $j$ and can be calculated as follows:

$c_{ij} = \frac{90 \cdot d_{ij}}{1000}$,

where $d_{ij}$ denotes the distance between $i$ and $j$.

We have two options to calculate $c_{ij}$ and assign the data to the GAMSPy parameter:

1. Python assignment - calculation in Python, e.g., using Pandas and `<parameter name>.setRecords()`
2. GAMSPy assignment - calculation in GAMSPy

#### Python Assignment

In [ ]:
cost = freight_cost * distances / 1000
cost

In [ ]:
c.setRecords(cost.reset_index())
c.records

#### GAMSPy Assignment

For the direct assignment we need to declare a new ``Parameter`` denoting the distances between $i$ and $j$.


In [ ]:
d = Parameter(
    container=m,
    name="d",
    domain=[i, j],
    description="distance between plant i and market j",
    records=distances.reset_index(),
)
d.records

In [ ]:
c[i, j] = freight_cost * d[i, j] / 1000
c.records

Further information on the usage of parameters can be found in our [parameter section](https://gamspy.readthedocs.io/en/latest/user/basics/parameter.html) of the user guide.

### Variables

GAMSPy variables are declared using `Variable`. Each ``Variable`` is assigned a name, a domain if necessary, a type, and, optionally, a description.

In [ ]:
x = Variable(
    container=m,
    name="x",
    domain=[i, j],
    type="Positive",
    description="amount of commodity to ship from plant i to market j",
)

This statement results in the declaration of a shipment variable for each (i,j) pair.

More information on variables can be found in the [variable section](https://gamspy.readthedocs.io/en/latest/user/basics/variable.html) of our user guide.

## Equations
A GAMSPy ``Equation`` must be declared and defined in two separate statements. The format of the declaration is the same as for other GAMSPy symbols. First comes the keyword, `Equation` in this case, followed by the name, domain and text. The transportation problem has two constraints:

Supply: observe supply limit at plant $i$: $\sum_j x_{ij} \le a_i \: \forall i$

Demand: satisfy demand at market $j$: $\sum_i x_{ij} \ge b_j \: \forall j$

In [ ]:
supply = Equation(
    container=m, name="supply", domain=i, description="observe supply limit at plant i"
)
demand = Equation(
    container=m, name="demand", domain=j, description="satisfy demand at market j"
)

The components of an ``Equation`` definition are:
1. The Python variable of the ``Equation`` being defined
2. The domain (optional)
3. Domain restricting conditions (optional)
4. A `=` sign
5. Left hand side expression
6. Relational operator (`==`, `<=`, `>=`)
7. The right hand side expression.

The ``Equation`` definition for the supply constraint of the transportation problem is implemented as follows:

In [ ]:
supply[i] = Sum(j, x[i, j]) <= a[i]

Using the same logic as above, we can define the demand equation as follows:

In [ ]:
demand[j] = Sum(i, x[i, j]) >= b[j]

More information on equations is given in the [equation section](https://gamspy.readthedocs.io/en/latest/user/basics/equation.html) of our user guide.

## Objective
The objective function of a GAMSPy ``Model`` does not require a separate ``Equation`` declaration. You can assign the objective expression to a Python variable or use it directly in the ``Model()`` statement of the [next section](#model).


In [ ]:
obj = Sum((i, j), c[i, j] * x[i, j])

## Model

A GAMSPy `Model()` consolidates constraints, an objective function, a sense (minimize, maximize, and feasibility), and a problem type. It also possesses a name and is associated with a ``Container``.

To define our transportation problem as a GAMSPy ``Model``, we assign it to a Python variable, link it to our ``Container`` (populated with symbols and data), name it "transport", specify the equations, set the problem type as linear program (LP), specify the sense of the objective function (``Sense.MIN``), and point to the objective expression.

GAMSPy allows two alternatives to assign equations to a `Model`:
1. Using a list of equations,
2. Retrieving _all_ equations by calling `m.getEquations()`.

### Using a List of Equations
Using a list of equations is especially useful if you want to define multiple GAMSPy ``Model``s with a subset of the  equations in your ``Container``. For the transportation problem this can be done as follows:

In [ ]:
transport = Model(
    m,
    name="transport",
    equations=[supply, demand],
    problem="LP",
    sense=Sense.MIN,
    objective=obj,
)

### Retrieving all Equations
Using `m.getEquations()` is especially convenient if you want to include all equations of your ``Container`` to be associated with your model. For the transportation problem this can be done as follows:

In [ ]:
transport_2 = Model(
    m,
    name="transport2",
    equations=m.getEquations(),
    problem="LP",
    sense=Sense.MIN,
    objective=obj,
)

More information on the usage of a GAMSPy `Model` can be found in the [model section](https://gamspy.readthedocs.io/en/latest/user/basics/model.html) of our user guide.

## Solve

Upon defining the GAMSPy ``Model``, it's ready for being solved. The ``solve()`` statement triggers the generation of the specific model instance, creates suitable data structures for the solver, and invokes the solver. To view solver output in the console, the ``sys`` library can be used, passing the ``output=sys.stdout`` attribute to ``transport.solve()``.

**Note**: The following cell contains the two most important pieces to get a GAMSPy model being solved by NVIDIA cuOpt:
1. Disable solver validation via `gp.set_options({"SOLVER_VALIDATION": 0})` as the solver is manually "plugged into" GAMSPy
2. Choose cuOpt as solver with `transport.solve(solver="cuopt", ...)`

Furthermore it sets two cuOpt solver options:
1. The [LP solution method](https://docs.nvidia.com/cuopt/user-guide/latest/lp-milp-settings.html#method) to PDLP
2. The [crossover flag](https://docs.nvidia.com/cuopt/user-guide/latest/lp-milp-settings.html#crossover) to false

In [ ]:
import sys
import gamspy as gp

gp.set_options({"SOLVER_VALIDATION": 0})
transport.solve(solver="cuopt", solver_options=dict(method=1, crossover=0), output=sys.stdout)

## Retrieving Results
### Variable Values
The values of the variables in the solution can be retrieved using using ``<variable name>.records``. The level specifies the shipment quantities $x_{ij}$. Other variable attributes are the marginal values, lower and upper bounds, and the variable's scaling factor.

In [ ]:
x.records.set_index(["i", "j"])

### Objective Value
The optimal objective function value can be accessed by `<model name>.objective_value`.

In [ ]:
transport.objective_value

SPDX-FileCopyrightText: Copyright (c) 2025 NVIDIA CORPORATION & AFFILIATES. All rights reserved.

SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.